# 16.4 — BCG v08 prediction workflow (inspect intermediates)

Run this after Josh hands over the **v08-retrained** scANVI control / treated files.
Kernel: **`analysis`**. The predict/eval scripts switch envs themselves.

Do **not** log1p the mouse files first. They should be human ENSG with atlas-posed
`layers['counts']` (continuous, count-scale). This notebook:

1. Checks those files look like that (stop here if they do not)
2. Runs `predict_new_input.sh --model-set uncapped_v08_iid --posed-ensg`
3. Checks the aligned + predicted outputs
4. Puts the human unvaccinated ground truth on the same v08 axis
5. Scores control-mouse predictions vs real unvaccinated human

Each step writes files and prints numbers. If something is wrong, the failing check
is the one to look at with Josh — do not jump to eval.

`RUN_PREDICT` / `RUN_EVAL` start as **False**. Run the verify cells first, then flip
the flags.


In [ ]:
import os
import subprocess
import sys

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

REPO = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
D = os.path.join(REPO, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
V08_TXT = os.path.join(D, "gene_lists/hvg_pearson_residuals_a_uncapped_v08_genes.txt")
V08_H5AD = os.path.join(D, "hvg_pearson_residuals_a_uncapped_v08.h5ad")
PREDICT_SH = os.path.join(REPO, "scripts/predict_new_input.sh")
H5AD_TO_V07 = os.path.join(REPO, "scripts/h5ad_to_v07.py")
EVAL_PY = os.path.join(REPO, "scripts/eval_external_target.py")
CELLOT_PY = os.path.expanduser("~/.conda/envs/CellOT/bin/python")
AEDIR = os.path.join(REPO, "cellot/cellot_gpu/results/hvg_pearson_residuals_a_uncapped_v08_iid/scgen")

# --- fill these when the v08 scANVI files arrive ---
MOUSE_CTRL = "/path/to/bcg_control_scanvi_v08.h5ad"   # unvaccinated / day-0
MOUSE_TRT = "/path/to/bcg_treated_scanvi_v08.h5ad"    # vaccinated / day-28
HUMAN_CTRL = "/path/to/bcg_human_unvax.h5ad"          # raw integer counts, ENSG names

TAG_CTRL = "bcg_ctrl_a2"
TAG_TRT = "bcg_trt_a2"

RUN_PREDICT = False
RUN_EVAL = False

v08_genes = [g.strip() for g in open(V08_TXT) if g.strip()]
assert len(v08_genes) == 1000, len(v08_genes)
print("v08 Pearson genes:", len(v08_genes), "from", V08_TXT)
for label, p in [("MOUSE_CTRL", MOUSE_CTRL), ("MOUSE_TRT", MOUSE_TRT),
                 ("HUMAN_CTRL", HUMAN_CTRL), ("predict.sh", PREDICT_SH),
                 ("aedir", AEDIR)]:
    print(("OK " if os.path.exists(p) else "MISSING ") + f"{label}: {p}")


## 1. Verify the batch-corrected mouse files

What we expect for `--posed-ensg`:

| Check | Pass |
|---|---|
| `.var_names` | human `ENSG…` |
| `layers['counts']` | present, **continuous** (not integers), count-scale (max typically ≫ 15) |
| `layers['counts_original']` | optional; if present, **integers** (raw UMIs) |
| row sums of posed vs original | roughly the same per cell (library-size rescale) |
| overlap with v08 Pearson | Josh saw **996 / 1000**; Aug-5 files were only 498 / 1000 |
| `.X` | ignore (often leftover log-like) |

If `layers['counts']` is already integer, this is probably `counts_original` (attempt 1),
not the posed decode. If max(`counts`) < 15, someone may have already log1p'd — **do not**
send that through the script.


In [ ]:
def _dense(X):
    return X.toarray() if sp_sparse.issparse(X) else np.asarray(X)


def _strip(g):
    g = str(g)
    return g.split(".")[0] if g.startswith("ENS") and "." in g else g


def verify_posed_mouse(path, label, v08, expect_min_overlap=900):
    print("=" * 72)
    print(label, path)
    if not os.path.exists(path) or path.startswith("/path/to/"):
        print("  SKIP: path not set or file missing")
        return None
    a = sc.read_h5ad(path)
    names = [_strip(g) for g in a.var_names.astype(str)]
    n_ensg = sum(g.startswith("ENSG") for g in names)
    layers = list(a.layers.keys()) if a.layers else []
    print(f"  shape {a.shape}  layers {layers}")
    print(f"  obs sample {list(a.obs.columns)[:12]}")
    print(f"  var sample {names[:5]}")
    print(f"  ENSG in var_names: {n_ensg}/{a.n_vars}")
    if "cell_type" in a.obs:
        print("  cell_type", a.obs["cell_type"].astype(str).value_counts().to_dict())

    checks = []
    checks.append(("human ENSG names", n_ensg >= max(10, int(0.8 * a.n_vars))))
    checks.append(("layers has counts", "counts" in a.layers))

    if "counts" in a.layers:
        C = _dense(a.layers["counts"]).astype(np.float64)
        frac_int = float(np.mean(np.isclose(C, np.round(C), atol=1e-5)))
        print(f"  layers['counts']: min={C.min():.4g} max={C.max():.4g} mean={C.mean():.4g}  frac_integer={frac_int:.4f}")
        checks.append(("counts are continuous (posed, not UMIs)", frac_int < 0.05))
        checks.append(("counts look count-scale, not already log1p", float(C.max()) > 15))
        if "counts_original" in a.layers:
            O = _dense(a.layers["counts_original"]).astype(np.float64)
            o_int = float(np.mean(np.isclose(O, np.round(O), atol=1e-5)))
            lib_c = C.sum(axis=1)
            lib_o = O.sum(axis=1)
            rel = np.abs(lib_c - lib_o) / np.maximum(lib_o, 1e-6)
            print(f"  layers['counts_original']: max={O.max():.4g} frac_integer={o_int:.4f}")
            print(f"  per-cell library |posed-orig|/orig: median={np.median(rel):.3g} max={rel.max():.3g}")
            checks.append(("counts_original are integers", o_int > 0.99))
            checks.append(("posed library ≈ original UMI total", float(np.median(rel)) < 0.05))
    else:
        checks.append(("layers has counts", False))

    X = _dense(a.X)
    print(f"  .X (ignore): min={X.min():.4g} max={X.max():.4g} mean={X.mean():.4g}")

    overlap = len(set(names) & set(v08))
    print(f"  overlap with v08 Pearson: {overlap} / 1000")
    checks.append((f"v08 overlap ≥ {expect_min_overlap}", overlap >= expect_min_overlap))
    if overlap < 700:
        print("  WARNING: this looks like the old v07/685-gene export, not the v08 retrain.")

    print("\n  checklist:")
    ok = True
    for name, passed in checks:
        print(("    PASS  " if passed else "    FAIL  ") + name)
        ok = ok and passed
    print("  overall:", "READY for --posed-ensg" if ok else "NOT READY — fix with Josh before predict")
    return {"adata": a, "overlap": overlap, "ok": ok, "names": names}


In [ ]:
ctrl_info = verify_posed_mouse(MOUSE_CTRL, "control / unvaccinated", v08_genes)
trt_info = verify_posed_mouse(MOUSE_TRT, "treated / vaccinated", v08_genes)


## 2. Predict human cells

Same command the mentor can run in a shell. Flip `RUN_PREDICT = True` in the paths cell
after section 1 is all PASS.

The script still does `normalize_total` + `log1p` once. `--posed-ensg` skips the
integer check and the mouse→human hop.


In [ ]:
def predict_cmd(mouse_path, tag):
    return [
        "bash", PREDICT_SH,
        "--model-set", "uncapped_v08_iid",
        "--posed-ensg",
        mouse_path,
        tag,
    ]


for mouse_path, tag in [(MOUSE_CTRL, TAG_CTRL), (MOUSE_TRT, TAG_TRT)]:
    cmd = predict_cmd(mouse_path, tag)
    print("\n$", " ".join(cmd))
    if not RUN_PREDICT:
        print("  (dry run — set RUN_PREDICT = True to execute)")
        continue
    if not os.path.exists(mouse_path) or mouse_path.startswith("/path/to/"):
        raise FileNotFoundError(mouse_path)
    proc = subprocess.run(cmd, cwd=REPO, check=False)
    print("  exit", proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError(f"predict_new_input.sh failed for {tag}")


## 3. Verify aligned + predicted outputs

After a successful run, Phase 1 prints `coverage N/1000`. Confirm here that the
aligned matrix is **exactly** the v08 gene order (the model will refuse otherwise).


In [ ]:
def aligned_paths(tag):
    flav = "pearson_residuals_uncapped_v08_iid"
    return {
        "aligned": os.path.join(D, f"{tag}_aligned_{flav}.h5ad"),
        "aligned07": os.path.join(D, f"{tag}_aligned_{flav}_anndata07.h5ad"),
        "pred_impact": os.path.join(D, f"{tag}_predicted_human_via_impact_cellot_{flav}.h5ad"),
        "pred_scgen": os.path.join(D, f"{tag}_predicted_human_via_scgen_{flav}.h5ad"),
    }


def verify_aligned(path, v08, label):
    print("=" * 72)
    print(label, path)
    if not os.path.exists(path):
        print("  MISSING (run section 2 with RUN_PREDICT = True)")
        return None
    a = sc.read_h5ad(path)
    names = [_strip(g) for g in a.var_names.astype(str)]
    X = _dense(a.X)
    print(f"  shape {a.shape}")
    print(f"  .X min={X.min():.4g} max={X.max():.4g} mean={X.mean():.4g}")
    print(f"  n_genes==1000: {a.n_vars == 1000}")
    print(f"  gene order == v08 list: {names == list(v08)}")
    if names != list(v08):
        print("  first mismatch:", next(
            ((i, names[i], v08[i]) for i in range(min(len(names), len(v08))) if names[i] != v08[i]),
            None,
        ))
    # log1p(CP10k) typically lives in ~0–8; posed raw counts were tens–hundreds
    print(f"  looks log1p-scale (max < 15): {float(X.max()) < 15}")
    return a


for tag in (TAG_CTRL, TAG_TRT):
    paths = aligned_paths(tag)
    print(f"\n### {tag} files on disk")
    for k, p in paths.items():
        print(("  OK " if os.path.exists(p) else "  -- ") + f"{k}: {p}")
    verify_aligned(paths["aligned"], v08_genes, f"{tag} aligned")
    if os.path.exists(paths["pred_impact"]):
        verify_aligned(paths["pred_impact"], v08_genes, f"{tag} IMPACT pred")


## 4. Human unvaccinated ground truth → same v08 axis

Human file must be **raw integer counts** with ENSG in `.var_names`.
`predict_new_input.sh` is mouse-only, so this cell does the projection + Scanpy.


In [ ]:
HUMAN_OUT = os.path.join(D, "bcg_human_unvax_target_pearson_residuals_uncapped_v08_iid.h5ad")
HUMAN_OUT07 = os.path.join(D, "bcg_human_unvax_target_pearson_residuals_uncapped_v08_iid_anndata07.h5ad")

if not os.path.exists(HUMAN_CTRL) or HUMAN_CTRL.startswith("/path/to/"):
    print("HUMAN_CTRL not set — skip. Fill the path and re-run this cell.")
else:
    src = sc.read_h5ad(HUMAN_CTRL)
    if "counts" in src.layers:
        src.X = src.layers["counts"].astype(np.float32)
    X = _dense(src.X).astype(np.float32)
    xs = X[:50].ravel()
    print("human source", src.shape, "var sample", list(src.var_names[:5]))
    print("integer counts?", bool(np.allclose(xs, np.round(xs))))
    if not np.allclose(xs, np.round(xs)):
        raise ValueError("human target must be raw integer counts")
    pos = {_strip(g): i for i, g in enumerate(src.var_names.astype(str))}
    Xn = np.zeros((src.n_obs, len(v08_genes)), dtype=np.float32)
    hit = 0
    for j, g in enumerate(v08_genes):
        i = pos.get(_strip(g))
        if i is not None:
            Xn[:, j] = X[:, i]
            hit += 1
    a = ad.AnnData(
        X=Xn,
        obs=src.obs.copy(),
        var=pd.DataFrame(index=pd.Index(v08_genes, name="ensg")),
    )
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    a.write_h5ad(HUMAN_OUT)
    print(f"human coverage {hit}/{len(v08_genes)} ({100*hit/len(v08_genes):.1f}%) -> {HUMAN_OUT}")
    if hit < 900:
        print("WARNING: human coverage is low — scoring will be dominated by zeros.")


## 5. Rewrite human target for the CellOT env (anndata 0.7)

The `_anndata07` suffix is the **file format**, not model v07.


In [ ]:
if not os.path.exists(HUMAN_OUT):
    print("HUMAN_OUT missing — run section 4 first")
else:
    cmd = [CELLOT_PY, H5AD_TO_V07, HUMAN_OUT, HUMAN_OUT07]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("wrote", HUMAN_OUT07)


## 6. Score control-mouse predictions vs real unvaccinated human

`--aedir` is the v08 IID **scgen** folder even when the prediction is IMPACT
(the OT map has no decoder of its own).

Read **`model_over_floor`** and **`r2_model_dec`**. Do not rank on
`frac_gap_closed_decoded` tonight.

Only score treated-mouse preds against a vaccinated human file (repeat 4–5 on that
object). Do not score treated mouse vs unvaccinated human if the question is the
species map.


In [ ]:
def eval_cmd(pred, tag):
    src = aligned_paths(TAG_CTRL)["aligned07"]
    return [
        CELLOT_PY, EVAL_PY,
        "--pred", pred,
        "--target", HUMAN_OUT07,
        "--source", src,
        "--aedir", AEDIR,
        "--tag", tag,
    ]


jobs = [
    (aligned_paths(TAG_CTRL)["pred_impact"], "bcg_ctrl_a2_impact_v08iid"),
    (aligned_paths(TAG_CTRL)["pred_scgen"], "bcg_ctrl_a2_scgen_v08iid"),
]
for pred, tag in jobs:
    cmd = eval_cmd(pred, tag)
    print("\n$", " ".join(cmd))
    if not RUN_EVAL:
        print("  (dry run — set RUN_EVAL = True to execute)")
        continue
    for p in (pred, HUMAN_OUT07, aligned_paths(TAG_CTRL)["aligned07"]):
        if not os.path.exists(p):
            raise FileNotFoundError(p)
    subprocess.run(cmd, cwd=REPO, check=True)


## 7. Read the scorecards


In [ ]:
EVAL_ROOT = os.path.join(REPO, "results/external_eval")
keep = [
    "tag", "n_cells", "model_over_floor", "r2_model_dec", "r2_model",
    "mmd_model", "mmd_ae_recon_floor", "mmd_decoded_ceiling",
    "frac_gap_closed_decoded", "mean_js",
]
rows = []
for tag in ("bcg_ctrl_a2_impact_v08iid", "bcg_ctrl_a2_scgen_v08iid"):
    p = os.path.join(EVAL_ROOT, tag, "external_target_metrics.csv")
    print(("OK " if os.path.exists(p) else "MISSING ") + p)
    if os.path.exists(p):
        df = pd.read_csv(p)
        df["tag"] = tag
        rows.append(df)
if rows:
    out = pd.concat(rows, ignore_index=True)
    cols = [c for c in keep if c in out.columns]
    print(out[cols].to_string(index=False))
